In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
project_dir = "/content/drive/MyDrive/Multiome_UMAP_Project/notebooks/analysis"
os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)


In [3]:
!pip install muon anndata scanpy mudata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.7/293.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.4/276.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 3.6 MB/s eta 0:00:00


In [4]:
# ==========================================
# Toy Multiome Dataset for Coupled UMAP
# ==========================================
import muon as mu
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.utils.extmath import randomized_svd

# Set random seed for reproducibility
np.random.seed(42)

# -------------------------------
# 1. Define dimensions
# -------------------------------
n_cells = 500
n_genes = 300
n_peaks = 1000

# -------------------------------
# 2. Create base random matrices
# -------------------------------
# RNA counts: Poisson distributed
X_rna = np.random.poisson(1.5, size=(n_cells, n_genes))

# ATAC accessibility: sparse binary matrix
X_atac = np.random.binomial(1, 0.05, size=(n_cells, n_peaks))

# -------------------------------
# 3. Create fake biological clusters
# -------------------------------
clusters = np.random.choice(["T", "B", "Mono"], n_cells)
for i, cell_type in enumerate(["T", "B", "Mono"]):
    mask = clusters == cell_type
    # Add modality-specific structure to RNA
    X_rna[mask, i*100:(i+1)*100] += np.random.poisson(2, size=(mask.sum(), 100))
    # Add structure to ATAC (slightly different regions)
    X_atac[mask, i*300:(i+1)*300] += np.random.binomial(1, 0.2, size=(mask.sum(), 300))

# -------------------------------
# 4. Wrap into AnnData objects
# -------------------------------
adata_rna = sc.AnnData(X=X_rna)
adata_rna.var_names = [f"gene_{i}" for i in range(n_genes)]
adata_rna.obs_names = [f"cell_{i}" for i in range(n_cells)]
adata_rna.obs["cell_type"] = clusters

adata_atac = sc.AnnData(X=X_atac)
adata_atac.var_names = [f"peak_{i}" for i in range(n_peaks)]
adata_atac.obs_names = adata_rna.obs_names
adata_atac.obs["cell_type"] = clusters

# -------------------------------
# 5. Combine modalities
# -------------------------------
mdata = mu.MuData({"rna": adata_rna, "atac": adata_atac})

print("✅ Created synthetic MuData object:")
print(mdata)

# -------------------------------
# 6. Preprocessing RNA
# -------------------------------
print("\n🔹 Preprocessing RNA modality...")
sc.pp.normalize_total(mdata.mod["rna"], target_sum=1e4)
sc.pp.log1p(mdata.mod["rna"])
sc.pp.highly_variable_genes(mdata.mod["rna"], flavor="seurat", n_top_genes=200)
mdata.mod["rna"] = mdata.mod["rna"][:, mdata.mod["rna"].var.highly_variable]
sc.pp.scale(mdata.mod["rna"], max_value=10)
sc.tl.pca(mdata.mod["rna"], n_comps=30)
print("RNA preprocessing complete.")

# -------------------------------
# 7. Preprocessing ATAC (TF-IDF + LSI)
# -------------------------------
print("\n🔹 Preprocessing ATAC modality...")

X = mdata.mod["atac"].X.astype(float)
# Term Frequency (TF)
tf = X / np.clip(X.sum(axis=1, keepdims=True), 1, None)
# Inverse Document Frequency (IDF)
idf = np.log(1 + X.shape[0] / np.clip((X > 0).sum(axis=0), 1, None))
X_tfidf = tf * idf

# Latent Semantic Indexing (SVD)
U, S, VT = randomized_svd(X_tfidf, n_components=30, random_state=42)
mdata.mod["atac"].obsm["X_lsi"] = U * S

print("ATAC preprocessing complete.")

# -------------------------------
# 8. Save processed data
# -------------------------------
mdata.write_h5mu("toy_multiome_preprocessed.h5mu")
print("\n💾 Saved toy dataset as 'toy_multiome_preprocessed.h5mu'")

# -------------------------------
# 9. Optional: Inspect embeddings
# -------------------------------
print("\nRNA PCA shape:", mdata.mod["rna"].obsm["X_pca"].shape)
print("ATAC LSI shape:", mdata.mod["atac"].obsm["X_lsi"].shape)
print("\nCell types:", mdata.mod["rna"].obs['cell_type'].value_counts())


/usr/local/lib/python3.12/dist-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/usr/local/lib/python3.12/dist-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
/usr/local/lib/python3.12/dist-packages/scanpy/preprocessing/_scale.py:309: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adat

✅ Created synthetic MuData object:
MuData object with n_obs × n_vars = 500 × 1300
  2 modalities
    rna:	500 x 300
      obs:	'cell_type'
    atac:	500 x 1000
      obs:	'cell_type'

🔹 Preprocessing RNA modality...
RNA preprocessing complete.

🔹 Preprocessing ATAC modality...
ATAC preprocessing complete.

💾 Saved toy dataset as 'toy_multiome_preprocessed.h5mu'

RNA PCA shape: (500, 30)
ATAC LSI shape: (500, 30)

Cell types: cell_type
Mono    182
B       159
T       159
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/usr/local/lib/python3.12/dist-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
